# AGILAB data and pipeline-graph launch in Colab

This notebook shows two built-in AGILAB paths from Google Colab:

- `execution_pandas_project` for a data-worker-style workload.
- `uav_relay_queue_project` for a DAG-shaped pipeline with topology and queue artifacts.

It clones the AGILAB repository, installs it in editable mode, installs
the worker the first time, and runs both paths locally through `AgiEnv`
and `AGI.run(...)`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThalesGroup/agilab/blob/main/examples/notebook_quickstart/agi_core_colab_data_dag.ipynb)


In [ ]:
# Simple install if you only want the published wheel:
# !python -m pip install -q uv agilab
#
# This notebook uses the source checkout below because it runs built-in
# data and DAG projects from the current repository state. It also forces
# AGI_CLUSTER_ENABLED=0 because this Colab path is local-only.
!if [ -d /content/agilab/.git ]; then cd /content/agilab && git pull --ff-only; else rm -rf /content/agilab && git clone --depth 1 https://github.com/ThalesGroup/agilab.git /content/agilab; fi
!python -m pip uninstall -y agilab agi-cluster agi-node agi-env >/dev/null 2>&1 || true
!python -m pip install -q uv
!pip install -q -e /content/agilab/src/agilab/core/agi-env -e /content/agilab/src/agilab/core/agi-node -e /content/agilab/src/agilab/core/agi-cluster -e /content/agilab


In [ ]:
from pathlib import Path

from agilab.notebook_colab_support import bootstrap_colab_core, install_if_needed

ctx = bootstrap_colab_core("/content/agilab")
AGI = ctx.AGI
AgiEnv = ctx.AgiEnv
APPS_PATH = ctx.apps_path

async def run_app(app_name: str):
    app_env = AgiEnv(apps_path=APPS_PATH, app=app_name, verbose=0)
    ctx.ensure_env_core_packages(app_env)
    await install_if_needed(AGI, app_env, scheduler="127.0.0.1", workers={"127.0.0.1": 1}, modes_enabled=0)
    result = await AGI.run(
        app_env,
        scheduler="127.0.0.1",
        workers={"127.0.0.1": 1},
        mode=0,
    )
    log_root = Path.home() / "log" / "execute" / app_env.target
    print(f"\n=== {app_name} ===")
    print("Target:", app_env.target)
    print("Log root:", log_root)
    if log_root.exists():
        print("Files:", sorted(path.name for path in log_root.iterdir())[:20])
    return result


In [ ]:
data_result = await run_app("execution_pandas_project")
data_result


In [ ]:
dag_result = await run_app("uav_relay_queue_project")
dag_result
